# 03 · Segmentation check

**Question.** Are there distinct player profiles in the behaviour data?
**Answer.** No usable structure. K-means does no better on the real data than on the same data with shuffled columns, and the variables are almost independent: the dataset looks simulated. Three descriptive groups are kept to illustrate the method, not as findings.
**Method.** K-means on standardised variables, compared with a shuffled-columns baseline, and stability across seeds (adjusted Rand index). Code: `game_players.segmentation`.

In [1]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

from game_players import config, segmentation
from game_players.database import connect

con = connect()
df = con.execute("SELECT * FROM gaming_behavior").df()
con.close()

all_vars = ["PlayTimeHours", "SessionsPerWeek", "AvgSessionDurationMinutes",
            "PlayerLevel", "AchievementsUnlocked", "InGamePurchases"]

## 1. Variables

In [2]:
print(df[all_vars].describe().round(2))
print(df[all_vars].corr().round(2))
print(df["InGamePurchases"].value_counts())

       PlayTimeHours  SessionsPerWeek  AvgSessionDurationMinutes  PlayerLevel  \
count       40034.00         40034.00                   40034.00     40034.00   
mean           12.02             9.47                      94.79        49.66   
std             6.91             5.76                      49.01        28.59   
min             0.00             0.00                      10.00         1.00   
25%             6.07             4.00                      52.00        25.00   
50%            12.01             9.00                      95.00        49.00   
75%            17.96            14.00                     137.00        74.00   
max            24.00            19.00                     179.00        99.00   

       AchievementsUnlocked  InGamePurchases  
count              40034.00          40034.0  
mean                  24.53              0.2  
std                   14.43              0.4  
min                    0.00              0.0  
25%                   12.00        

Correlations are close to zero and the distributions are flat: a first sign of simulated data. `InGamePurchases` is binary (20% of payers).

## 2. First attempt: six variables

In [3]:
X6, _ = segmentation.standardize(df, all_vars)
segmentation.scan(X6, range(2, 8))

,k,inertia,silhouette
0,2,200165,0.237
1,3,176018,0.160
2,4,161546,0.144
3,5,149694,0.144
4,6,141010,0.142
5,7,133280,0.143


In [4]:
ari = adjusted_rand_score(segmentation.kmeans(X6, 5, 42).labels_, segmentation.kmeans(X6, 5, 7).labels_)
print(f"Agreement of two seeds with 5 clusters (ARI): {ari:.3f}")

Agreement of two seeds with 5 clusters (ARI): 0.463


The silhouette is low whatever k, and 5 clusters change with the seed: weak structure.

## 3. Real data against shuffled data

Four variables, one per behaviour (`config.SEGMENT_VARIABLES`). Shuffling each column destroys any relation between variables: if the real silhouette is not clearly higher, the clusters are no more than noise.

In [5]:
X4, _ = segmentation.standardize(df, config.SEGMENT_VARIABLES)
segmentation.structure_check(X4)

,k,silhouette_real,silhouette_shuffled,ari_between_seeds
0,2,0.338,0.338,1.000
1,3,0.255,0.256,1.000
2,4,0.247,0.245,0.399
3,5,0.258,0.262,0.513
4,6,0.255,0.260,0.588


Real and shuffled silhouettes are the same: the clusters come from the shape of each variable alone. k = 2 is just payers vs non-payers; k = 2 and 3 are stable across seeds, larger k are not. k = 3 is kept as a readable description.

## 4. Three descriptive groups

In [6]:
seg = segmentation.fit_segments(df)
print(seg.groupby("segment")[config.SEGMENT_VARIABLES].mean().round(2))
print(seg["segment"].value_counts(normalize=True).round(3))
pd.crosstab(seg["segment"], seg["EngagementLevel"], normalize="index").round(2)

                          SessionsPerWeek  AvgSessionDurationMinutes  \
segment                                                                
Highly active non-payers            14.52                      94.75   
Low-activity non-payers              4.51                      94.98   
Payers                               9.53                      94.49   

                          PlayerLevel  InGamePurchases  
segment                                                 
Highly active non-payers        49.64              0.0  
Low-activity non-payers         49.48              0.0  
Payers                          50.03              1.0  
segment
Low-activity non-payers     0.404
Highly active non-payers    0.395
Payers                      0.201
Name: proportion, dtype: float64


EngagementLevel,High,Low,Medium
segment,,,
Highly active non-payers,0.47,0.09,0.45
Low-activity non-payers,0.05,0.43,0.52
Payers,0.27,0.25,0.48


## Conclusion

| Group | Share | Profile |
|---|---|---|
| Highly active non-payers | about 39% | many sessions, never paid |
| Low-activity non-payers | about 40% | few sessions, never paid |
| Payers | about 20% | paid at least once |

The groups split payment and session frequency, the two variables whose shape is not flat. The dataset's own engagement label follows sessions per week, as expected. On real player data, the same check (real vs shuffled, stability across seeds) would tell whether segments can drive targeted actions; here they cannot.